In [1]:
from pathlib import Path
import sys
import os
import subprocess
import importlib
import pandas as pd

# ここだけ自分の環境に合わせて変更してください
# 例1: Windows側に置いている場合
PROJECT_ROOT = Path("/home/shu/py_venvs/py2/rena_pj1")

# 例2: WSLのhome側に置いている場合
# PROJECT_ROOT = Path("/home/shu/核酸AIプロジェクト")

SRC_DIR = PROJECT_ROOT / "src"
DB_DIR = PROJECT_ROOT / "データベース"
DATA_DIR = DB_DIR / "data"

DATA_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(SRC_DIR))
os.chdir(SRC_DIR)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR     :", SRC_DIR)
print("DB_DIR      :", DB_DIR)
print("DATA_DIR    :", DATA_DIR)

PROJECT_ROOT: /home/shu/py_venvs/py2/rena_pj1
SRC_DIR     : /home/shu/py_venvs/py2/rena_pj1/src
DB_DIR      : /home/shu/py_venvs/py2/rena_pj1/データベース
DATA_DIR    : /home/shu/py_venvs/py2/rena_pj1/データベース/data


In [2]:
required_files = [
    SRC_DIR / "constants.xlsx",
    SRC_DIR / "compound.py",
    SRC_DIR / "oligomer.py",
    SRC_DIR / "peptide.py",
    SRC_DIR / "calc_to_make_db02.py",
    SRC_DIR / "calc_to_make_db07.py",
    SRC_DIR / "calc_to_make_db08.py",
    SRC_DIR / "calc_to_make_db12.py",
    SRC_DIR / "calc_to_make_db17.py",
    DB_DIR / "02_ss_oligonucleotide.xlsx",
    DB_DIR / "07_sequence_baseline.xlsx",
    DB_DIR / "08_peptide.xlsx",
    DB_DIR / "12_ss_oligonucleotide_forPB.xlsx",
    DB_DIR / "17_sequence_baseline_forPB.xlsx",
]

check_df = pd.DataFrame({
    "file": [str(p) for p in required_files],
    "exists": [p.exists() for p in required_files],
})

check_df

,file,exists
0,/home/shu/py_venvs/py2/rena_pj1/src/constants....,True
1,/home/shu/py_venvs/py2/rena_pj1/src/compound.py,True
2,/home/shu/py_venvs/py2/rena_pj1/src/oligomer.py,True
3,/home/shu/py_venvs/py2/rena_pj1/src/peptide.py,True
4,/home/shu/py_venvs/py2/rena_pj1/src/calc_to_ma...,True
5,/home/shu/py_venvs/py2/rena_pj1/src/calc_to_ma...,True
6,/home/shu/py_venvs/py2/rena_pj1/src/calc_to_ma...,True
7,/home/shu/py_venvs/py2/rena_pj1/src/calc_to_ma...,True
8,/home/shu/py_venvs/py2/rena_pj1/src/calc_to_ma...,True
9,/home/shu/py_venvs/py2/rena_pj1/データベース/02_ss_o...,True


In [3]:
weight_df = pd.read_excel(SRC_DIR / "constants.xlsx", sheet_name="weight")
weight_df

,name,note,C,H,B,Cl,F,K,N,Na,O,P,S,Si,Unnamed: 14
0,mol_w,NaN,12.0107,1.007940,10.811000,35.453000,18.998403,39.098300,14.006700,22.989769,15.999400,30.973762,32.065000,28.085500,28.085500
1,exact,NaN,12.0000,1.007825,11.009305,34.968853,18.998403,38.963706,14.003074,22.989769,15.994915,30.973762,31.972071,27.976927,27.976927


In [4]:
input_02 = pd.read_excel(DB_DIR / "02_ss_oligonucleotide.xlsx", sheet_name="sequence").fillna("")

input_02[[
    "id_sequence",
    "name",
    "baseline_sequence",
    "modification_5",
    "sequence",
    "modification_3",
]].head()

,id_sequence,name,baseline_sequence,modification_5,sequence,modification_3
0,02-00-00,malat1-ASO,07-00-07 malat1-ASO,04-00-00 H,mC(L)^T(L)^A(L)^G^T^T^C^A^C^T^G^A^A^T(L)^G(L)^...,05-00-00 H
1,02-00-01,malat1-SO,07-00-08 malat1-SO,04-00-00 H,G(O)^C(O)^A(O)^uucagugaac^U(O)^A(O)^G(O),05-00-00 H
2,02-00-02,malat1-SO(5′-DMTr),07-00-08 malat1-SO,04-00-03 DMTr,G(O)^C(O)^A(O)^uucagugaac^U(O)^A(O)^G(O),05-00-00 H
3,02-00-03,malat1-SO(5′-C6Amine),07-00-08 malat1-SO,04-00-01 C6Amine,G(O)^C(O)^A(O)^uucagugaac^U(O)^A(O)^G(O),05-00-00 H
4,02-00-04,malat1-SO(5′-DBCO),07-00-08 malat1-SO,04-00-04 DBCO,G(O)^C(O)^A(O)^uucagugaac^U(O)^A(O)^G(O),05-00-00 H


In [5]:
import calc_to_make_db02

importlib.reload(calc_to_make_db02)

const02 = calc_to_make_db02.ConstantsForSequence(
    filename_constant=str(SRC_DIR / "constants.xlsx"),
    filename_sequence=str(DB_DIR / "02_ss_oligonucleotide.xlsx"),
)

# id_sequence が空でない行だけにする
input_02_valid = input_02[input_02["id_sequence"].astype(str).str.strip() != ""]

# 先頭1件だけ取り出す
one_row = input_02_valid.iloc[0].to_dict()

# 1件だけオリゴ核酸オブジェクトに変換して計算
one_oligo = calc_to_make_db02.RenaOligomer.read_renafile(
    constants=const02,
    **one_row
)

# 結果を表で見る
one_result = pd.Series(one_oligo.expand_for_summary()).to_frame("value")
one_result

/home/shu/py_venvs/py2/rena_pj1/src/oligomer.py:1208: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  self.sequence_table["linker"][-1:] = self.constants.dict_linkers["PS"]
/home/shu/py_venvs/py2/rena_pj1/src/oligomer.py:1208: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behave

,value
ID Sequence,02-00-00
Name,malat1-ASO
memo,
id_mod5,04-00-00
Modification_5,H
id_baseline,07-00-07
sequence_ns,mC(L)^T(L)^A(L)^G^T^T^C^A^C^T^G^A^A^T(L)^G(L)^...
id_mod3,05-00-00
Modification_3,H
Length,16


In [7]:
import calc_to_make_db02

importlib.reload(calc_to_make_db02)

calc_to_make_db02.run_all()

output_02 = DATA_DIR / "02_ss_oligonucleotide.csv"

df_02 = pd.read_csv(output_02, encoding="utf-8-sig")

df_02[[
    "ID Sequence",
    "Name",
    "Formula",
    "ExactMS",
    "Mol.W.",
    "Extinct",
    "nmol/OD",
    "µg/OD",
    "Sequence(OP)",
    "sequence_gd",
]].head()


>> Start   :     calc_flp     |


/home/shu/py_venvs/py2/rena_pj1/src/oligomer.py:1208: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  self.sequence_table["linker"][-1:] = self.constants.dict_linkers["PS"]
/home/shu/py_venvs/py2/rena_pj1/src/oligomer.py:1208: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behave

>> End     :     calc_flp     |  RAP   : 00′04″61


/home/shu/py_venvs/py2/rena_pj1/src/oligomer.py:1206: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  self.sequence_table["linker"][-1:] = self.constants.dict_linkers["PO"]
/home/shu/py_venvs/py2/rena_pj1/src/oligomer.py:1208: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behave

,ID Sequence,Name,Formula,ExactMS,Mol.W.,Extinct,nmol/OD,µg/OD,Sequence(OP),sequence_gd
0,02-00-00,malat1-ASO,C164H202N57O86P15S15,5289.506030,5293.270410,149220.0,6.701515,35.472929,y^q^x^G^T^T^C^A^C^T^G^A^A^q^z^y,5(L)^T(L)^A(L)^g^t^t^c^a^c^t^g^a^a^T(L)^G(L)^5(L)
1,02-00-01,malat1-SO,C159H201N62O103P15S6,5282.678483,5285.647270,167040.0,5.986590,31.643003,z^y^x^TTCAGTGAAC^q^x^z,G(M)^C(M)^A(M)^UUCAGUGAAC^U(M)^A(M)^G(M)
2,02-00-02,malat1-SO(5′-DMTr),C180H219N62O105P15S6,5584.809162,5588.013690,167040.0,5.986590,33.453147,z^y^x^TTCAGTGAAC^q^x^z,G(M)^C(M)^A(M)^UUCAGUGAAC^U(M)^A(M)^G(M)
3,02-00-03,malat1-SO(5′-C6Amine),C165H215N63O106P16S6,5461.749613,5464.801292,167040.0,5.986590,32.715525,z^y^x^TTCAGTGAAC^q^x^z,G(M)^C(M)^A(M)^UUCAGUGAAC^U(M)^A(M)^G(M)
4,02-00-04,malat1-SO(5′-DBCO),C184H228N64O108P16S6,5748.844242,5752.113312,167040.0,5.986590,34.435544,z^y^x^TTCAGTGAAC^q^x^z,G(M)^C(M)^A(M)^UUCAGUGAAC^U(M)^A(M)^G(M)


In [8]:
from calc_to_make_db02 import Sequence, SequenceForRena, ConstantsForSequence

def make_baseline_sequence_csv(input_xlsx, output_csv):
    """
    07_sequence_baseline.xlsx または 17_sequence_baseline_forPB.xlsx の
    converterシートを読み、sequence_ns / sequence_gd を作ってCSV出力する関数。
    """

    const = ConstantsForSequence(
        filename_constant=str(SRC_DIR / "constants.xlsx"),
        filename_sequence=str(DB_DIR / "02_ss_oligonucleotide.xlsx"),
    )

    df_sheet = pd.read_excel(input_xlsx, sheet_name="converter").fillna("")

    def make_baseline(line):
        if line["type_modification"] == "SO":
            baseline = Sequence.create_fm_sequence_ns(
                line["baseline"],
                const,
            ).get_complementary_baseline(
                rna=False,
                reverse=False,
            )
        else:
            baseline = line["baseline"]

        wing5 = int(line["wing5"]) if line["wing5"] != "" else 0
        wing3 = int(line["wing3"]) if line["wing3"] != "" else 0

        oligo = SequenceForRena.create_fm_baseline_as_gapmer(
            sequence_baseline=baseline,
            sugar_wing=line["sugar_wing"],
            sugar_gap=line["sugar_gap"],
            linker_wing=line["link_wing"],
            linker_gap=line["link_gap"],
            wing=(wing5, wing3),
            constants=const,
        )
        return oligo

    df_sheet["obj"] = df_sheet.apply(make_baseline, axis=1)
    df_sheet["sequence_ns"] = df_sheet["obj"].apply(lambda x: x.sequence_ns)
    df_sheet["sequence_gd"] = df_sheet["obj"].apply(lambda x: x.get_seq_gd())

    output_cols = [
        "ID", "Name", "note", "id_sequence", "baselinename", "baseline",
        "len", "type_modification", "sugar_wing", "sugar_gap",
        "link_wing", "link_gap", "wing5", "wing3",
        "sequence_ns", "sequence_gd", "手動調整？",
    ]

    output_cols = [c for c in output_cols if c in df_sheet.columns]

    df_for_output = df_sheet[output_cols]

    df_for_output.to_csv(output_csv, encoding="utf-8-sig", index=False)

    return df_for_output

In [9]:
df_07 = make_baseline_sequence_csv(
    input_xlsx=DB_DIR / "07_sequence_baseline.xlsx",
    output_csv=DATA_DIR / "calcd_DB07_baselinesequence.csv",
)

df_07.head()

ValueError: Worksheet named 'converter' not found

In [10]:
df_17 = make_baseline_sequence_csv(
    input_xlsx=DB_DIR / "17_sequence_baseline_forPB.xlsx",
    output_csv=DATA_DIR / "calcd_DB17_baselinesequence.csv",
)

df_17.head()

/home/shu/py_venvs/py2/rena_pj1/src/calc_to_make_db02.py:154: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  oligomer.sequence_table["sugar"][: wing[0]] = obj_sugar_wing
/home/shu/py_venvs/py2/rena_pj1/src/calc_to_make_db02.py:155: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always 

,ID,Name,note,id_sequence,baselinename,baseline,len,type_modification,sugar_wing,sugar_gap,link_wing,link_gap,wing5,wing3,sequence_ns,sequence_gd,手動調整？
0,17-00-01,Bace1(All-DNA),,16-00-03,Bace1,GTATTGCTGAGGA,13,ASO,DNA,DNA,PB,PB,2,3,GTATTGCTGAGGA,gtattgctgagga,
1,17-00-02,Bace1 gapmer,,16-00-03,Bace1,GTATTGCTGAGGA,13,ASO,LNA,DNA,PB,PB,2,3,GTATTGCTGAGGA,gtattgctgagga,
2,17-00-03,SYNGAP1-24(All-DNA),,16-00-04,SYNGAP1-24,AGAAATGGAGAGGGAGAG,18,ASO,DNA,DNA,PB,PB,0,0,AGAAATGGAGAGGGAGAG,agaaatggagagggagag,
3,17-00-04,SYNGAP1-24(Full MOE),,16-00-04,SYNGAP1-24,AGAAATGGAGAGGGAGAG,18,ASO,MOE,MOE,PB,PB,0,0,A(M)G(M)A(M)A(M)A(M)T(M)G(M)G(M)A(M)G(M)A(M)G(...,A(m)G(m)A(m)A(m)A(m)T(m)G(m)G(m)A(m)G(m)A(m)G(...,
4,17-00-05,SYNGAP1-5(All-DNA),,16-00-05,SYNGAP1-5,GACACCAGGGAGAGACAG,18,ASO,DNA,DNA,PB,PB,0,0,GACACCAGGGAGAGACAG,gacaccagggagagacag,


In [11]:
import calc_to_make_db12

importlib.reload(calc_to_make_db12)

calc_to_make_db12.run_all()

output_12 = DATA_DIR / "12_ss_oligonucleotide.csv"

df_12 = pd.read_csv(output_12, encoding="utf-8-sig")

df_12[[
    "ID Sequence",
    "Name",
    "Formula",
    "ExactMS",
    "Mol.W.",
    "Extinct",
    "nmol/OD",
    "µg/OD",
    "Sequence(OP)",
    "sequence_gd",
]].head()


>> Start   :     calc_flp     |
>> End     :     calc_flp     |  RAP   : 00′00″12


/home/shu/py_venvs/py2/rena_pj1/src/oligomer.py:1208: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  self.sequence_table["linker"][-1:] = self.constants.dict_linkers["PS"]
/home/shu/py_venvs/py2/rena_pj1/src/oligomer.py:1208: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behave

,ID Sequence,Name,Formula,ExactMS,Mol.W.,Extinct,nmol/OD,µg/OD,Sequence(OP),sequence_gd
0,12-00-00,"Bace1(All-DNA)(5′-H, 3′-H)",C129H197B12N51O65P12,4004.164564,4003.664324,131310.0,7.615566,30.490171,G*T*A*T*T*G*C*T*G*A*G*G*A,g*t*a*t*t*g*c*t*g*a*g*g*a
1,12-00-01,"Bace1 gapmer(5′-H, 3′-H)",C134H197B12N51O70P12,4144.139137,4143.714824,131310.0,7.615566,31.556735,z*q*A*T*T*G*C*T*G*A*z*z*x,G(L)*T(L)*a*t*t*g*c*t*g*a*G(L)*G(L)*A(L)
2,12-00-02,"SYNGAP1-24(All-DNA)(5′-H, 3′-H)",C180H269B17N87O82P17,5674.667516,5673.936514,211860.0,4.720098,26.781537,A*G*A*A*A*T*G*G*A*G*A*G*G*G*A*G*A*G,a*g*a*a*a*t*g*g*a*g*a*g*g*g*a*g*a*g
3,12-00-03,"SYNGAP1-24(Full MOE)(5′-H, 3′-H)",C234H377B17N87O118P17,7007.329546,7007.350234,211860.0,4.720098,33.075381,x*z*x*x*x*q*z*z*x*z*x*z*z*z*x*z*x*z,A(m)*G(m)*A(m)*A(m)*A(m)*T(m)*G(m)*G(m)*A(m)*G...
4,12-00-04,"SYNGAP1-5(All-DNA)(5′-H, 3′-H)",C176H268B17N82O82P17,5555.644321,5554.852274,196110.0,5.099179,28.325186,G*A*C*A*C*C*A*G*G*G*A*G*A*G*A*C*A*G,g*a*c*a*c*c*a*g*g*g*a*g*a*g*a*c*a*g


In [12]:
import calc_to_make_db08

importlib.reload(calc_to_make_db08)

# peptide sequence の計算
sheet_peptide = calc_to_make_db08.Sheet(calc_to_make_db08.FILENAME, "sequence")
sheet_peptide.calc_ms()
sheet_peptide.output_data()

# ligand の計算
sheet_ligand = calc_to_make_db08.Sheet(calc_to_make_db08.FILENAME, "ligand")
sheet_ligand.calc_ms()
sheet_ligand.output_liganddata()

output_08 = DATA_DIR / "08_peptide.csv"
output_ligand = DATA_DIR / "ligand.csv"

df_08 = pd.read_csv(output_08, encoding="utf-8-sig")

df_08[[
    "ID",
    "Name",
    "Length",
    "N_terminal",
    "Sequence",
    "C_terminal",
    "Formula",
    "Ex.MS",
    "Mol.W",
    "ε(280 nm)",
]].head()


>> Start   :     calc_ms      |
>> End     :     calc_ms      |  RAP   : 00′00″24

>> Start   :     calc_ms      |
>> End     :     calc_ms      |  RAP   : 00′00″17


,ID,Name,Length,N_terminal,Sequence,C_terminal,Formula,Ex.MS,Mol.W,ε(280 nm)
0,08-00-01,cSLS,15,H,ACSLSHSPQCGGGS(K-COCH2-Azide),OH,C57H90N22O22S2,1498.604145,1499.58870,250.0
1,08-00-02,Ang2,22,H,TFFYGGSRGKRNNFKTEEYGG(K-COCH2-Azide),OH,C116H168N36O35,2625.247281,2626.79532,2980.0
2,08-00-03,MTfp,14,H,DSSHAFTLDELRY(K-COCH2-Azide),OH,C76H113N23O26,1763.822711,1764.84892,1490.0
3,08-00-04,eGLP1,40,H,H(AiB)EGTFTSDVSSYLEEQAAKEFIAWLVKGGPSSGAPPPSC,NH2,C186H279N47O60S,4162.994611,4165.54936,7115.0
4,08-00-05,eNT,22,H,(K-COCH2-Azide)PPPAGSSPGLYENKPRRPYIL,OH,C114H178N34O31,2519.339725,2520.84232,2980.0
